# TabM + Periodic Embeddings + Swap-Noise DAE (5-Fold)

**Cấu hình chạy trên Kaggle:** Accelerator = GPU T4 x2, Internet = Off.
- Huấn luyện 5-Fold (`StratifiedKFold`, `seed=42`) trên `train.csv`.
- Kết quả xuất ra `/kaggle/working/`: `submission.csv`, `oof_kaggle_tabm_5fold.csv`, `sub_kaggle_tabm_5fold.csv`.


In [ ]:
"""Quy trình 5-Fold TabM kết hợp mã hóa tuần hoàn và bộ tự khử nhiễu DAE không giám sát."""

from __future__ import annotations

import math
import os
from pathlib import Path
import time

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import expit, logit
from sklearn.linear_model import Ridge
from sklearn.metrics import recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
import torch
from torch import nn
import torch.nn.functional as F

# Cấu hình huấn luyện (bật DRY_RUN=True để chạy thử nhanh trên 3.000 dòng)
DRY_RUN = os.environ.get("KAGGLE_DRY_RUN", "0") == "1"
SEED = 42
N_SPLITS = 3 if DRY_RUN else 5
TABM_K = 8 if DRY_RUN else 16
TABM_EPOCHS = 2 if DRY_RUN else 20
TABM_PATIENCE = 5
DAE_EPOCHS = 2 if DRY_RUN else 12
BATCH_SIZE = 256 if DRY_RUN else 2048
AUC_LOSS_WEIGHT = 0.25

ID_COLUMN = "id"
TARGET_COLUMN = "is_fraud"
NUMERIC_COLUMNS = [
    "amount_usd",
    "hours_since_last_txn",
    "txn_count_last_24h",
    "distance_from_home_km",
    "card_age_months",
    "customer_age",
    "account_balance_usd",
    "cvv_retry_count",
    "velocity_score",
    "time_of_day_hour",
    "day_of_week",
    "merchant_risk_score",
    "prior_disputes",
]
FLOAT_COLUMNS = [
    "amount_usd",
    "account_balance_usd",
    "hours_since_last_txn",
    "distance_from_home_km",
    "velocity_score",
    "merchant_risk_score",
]
BINARY_COLUMNS = [
    "is_foreign_transaction",
    "is_new_merchant",
    "used_vpn",
    "ip_country_mismatch",
    "billing_shipping_mismatch",
    "is_ai_generated_scam_attempt",
]
CATEGORICAL_COLUMNS = [
    "merchant_category",
    "card_type",
    "auth_method",
    "channel",
    "device_type",
]


def locate_data_dir() -> Path:
    candidates = [
        Path("/kaggle/input/playground-series-s6e11"),
        Path("/kaggle/input/competitions/playground-series-s6e11"),
        Path("dataset/playground-series-s6e11"),
        Path("../dataset/playground-series-s6e11"),
    ]
    for path in candidates:
        if (path / "train.csv").exists():
            return path
    # Tìm dự phòng trong /kaggle/input
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        for p in kaggle_root.rglob("train.csv"):
            if (p.parent / "test.csv").exists():
                return p.parent
    raise FileNotFoundError("Could not find train.csv / test.csv. Attach the playground-series-s6e11 dataset.")


def locate_output_dir() -> Path:
    custom = os.environ.get("KAGGLE_OUT_DIR")
    if custom:
        out = Path(custom)
        out.mkdir(parents=True, exist_ok=True)
        return out
    if Path("/kaggle/working").exists():
        return Path("/kaggle/working")
    out = Path("artifacts/kaggle_dryrun") if DRY_RUN else Path("artifacts/kaggle_gpu")
    out.mkdir(parents=True, exist_ok=True)
    return out


def safe_logit(p: np.ndarray | float) -> np.ndarray | float:
    return logit(np.clip(p, 1e-6, 1.0 - 1e-6))


def _leading_digits(int_arr: np.ndarray, k: int) -> np.ndarray:
    vals = np.maximum(int_arr, 0).astype(np.int64)
    out = np.zeros_like(vals)
    pos = vals > 0
    if not pos.any():
        return out
    v = vals[pos]
    digits = np.floor(np.log10(v.astype(np.float64))).astype(np.int64) + 1
    shift = np.maximum(digits - k, 0)
    pow10 = np.power(10, np.minimum(shift, 12), dtype=np.int64)
    out[pos] = v // pow10
    return out


def extract_keys(df: pd.DataFrame) -> tuple[dict[str, np.ndarray], dict[str, np.ndarray]]:
    keys, freqs = {}, {}
    for col in NUMERIC_COLUMNS:
        c100 = np.rint(df[col].to_numpy(dtype=np.float64) * 100.0).astype(np.int64)
        keys[f"{col}__exact"] = c100
        freqs[f"{col}__freq"] = c100
    for col in FLOAT_COLUMNS:
        c100 = keys[f"{col}__exact"]
        ip = np.floor_divide(c100, 100)
        keys[f"{col}__int"] = ip
        keys[f"{col}__r1"] = np.floor_divide(c100, 10)
        keys[f"{col}__dec2"] = np.mod(c100, 100)
        keys[f"{col}__cent1"] = np.mod(c100, 10)
        keys[f"{col}__cent2"] = np.mod(np.floor_divide(c100, 10), 10)
        keys[f"{col}__unit"] = np.mod(ip, 10)
        keys[f"{col}__tens"] = np.mod(np.floor_divide(ip, 10), 10)
        keys[f"{col}__lead1"] = _leading_digits(ip, 1)
        keys[f"{col}__lead2"] = _leading_digits(ip, 2)
        freqs[f"{col}__int_freq"] = ip
        if col in ("amount_usd", "account_balance_usd"):
            keys[f"{col}__int_mod100"] = np.mod(ip, 100)
    return keys, freqs


def _lookup_sorted(k_arr: np.ndarray, uniq: np.ndarray, vals: np.ndarray, default: float = 0.0) -> np.ndarray:
    idx = np.searchsorted(uniq, k_arr)
    idx_s = np.minimum(idx, uniq.size - 1)
    return np.where(uniq[idx_s] == k_arr, vals[idx_s], default).astype(np.float32)


def build_base_arrays(df: pd.DataFrame, cat_vocabs: dict[str, dict[str, int]] | None = None):
    """Tạo mảng đặc trưng số, đặc trưng nghiệp vụ, cờ nhị phân và One-Hot phân loại."""
    num = np.column_stack([df[c].to_numpy(dtype=np.float32) for c in NUMERIC_COLUMNS])
    bin_mat = np.column_stack([df[c].astype(np.float32).to_numpy() for c in BINARY_COLUMNS])

    amt = df["amount_usd"].to_numpy(dtype=np.float64)
    bal = df["account_balance_usd"].to_numpy(dtype=np.float64)
    hrs = df["hours_since_last_txn"].to_numpy(dtype=np.float64)
    dist = df["distance_from_home_km"].to_numpy(dtype=np.float64)
    hour = df["time_of_day_hour"].to_numpy(dtype=np.float64)
    dow = df["day_of_week"].to_numpy(dtype=np.float64)

    log_amt = np.log1p(np.maximum(amt, 0.0))
    log_bal = np.log1p(np.maximum(bal, 0.0))
    log_hrs = np.log1p(np.maximum(hrs, 0.0))
    log_dist = np.log1p(np.maximum(dist, 0.0))

    domain = np.column_stack([
        bin_mat.sum(axis=1),
        log_amt,
        log_bal,
        log_hrs,
        log_dist,
        log_amt - log_bal,
        amt / (np.maximum(bal, 0.0) + 1.0),
        (amt > bal).astype(np.float64),
        dist / (np.maximum(hrs, 0.0) + 0.1),
        np.sin(2.0 * np.pi * hour / 24.0),
        np.cos(2.0 * np.pi * hour / 24.0),
        np.sin(2.0 * np.pi * dow / 7.0),
        np.cos(2.0 * np.pi * dow / 7.0),
    ]).astype(np.float32)

    if cat_vocabs is None:
        cat_vocabs = {}
        for col in CATEGORICAL_COLUMNS:
            vals = sorted(df[col].dropna().astype("string").unique().tolist())
            cat_vocabs[col] = {v: i + 1 for i, v in enumerate(vals)}

    oh_parts = []
    for col in CATEGORICAL_COLUMNS:
        vocab = cat_vocabs[col]
        codes = df[col].astype("string").map(vocab).fillna(0).to_numpy(dtype=np.int64)
        oh = np.zeros((len(df), len(vocab) + 1), dtype=np.float32)
        oh[np.arange(len(df)), codes] = 1.0
        oh_parts.append(oh)
    oh_mat = np.concatenate(oh_parts, axis=1)
    return num, domain, bin_mat, oh_mat, cat_vocabs


def fit_transform_cross_residuals(
    num_tr: np.ndarray,
    oh_tr: np.ndarray,
    other_pairs: list[tuple[np.ndarray, np.ndarray]],
) -> tuple[np.ndarray, list[np.ndarray]]:
    """Tính phần dư hồi quy Ridge giữa các cột đầu vào để phát hiện điểm bất thường (không dùng nhãn)."""
    # Chỉ số các cột trong NUMERIC_COLUMNS:
    # 0:amount, 1:hours, 2:txn24, 3:dist, 4:card_age, 5:cust_age, 6:balance, 7:cvv, 8:velocity, 9:hour, 10:dow, 11:risk, 12:disputes
    X_tr_all = np.concatenate([
        np.log1p(np.maximum(num_tr[:, [0, 1, 3, 6]], 0.0)),
        num_tr[:, [2, 4, 5, 7, 8, 9, 10, 11, 12]],
        oh_tr,
    ], axis=1)
    mean = X_tr_all.mean(axis=0)
    std = np.where(X_tr_all.std(axis=0) < 1e-6, 1.0, X_tr_all.std(axis=0))
    X_tr_s = (X_tr_all - mean) / std

    X_others_s = []
    for n_o, oh_o in other_pairs:
        xo = np.concatenate([
            np.log1p(np.maximum(n_o[:, [0, 1, 3, 6]], 0.0)),
            n_o[:, [2, 4, 5, 7, 8, 9, 10, 11, 12]],
            oh_o,
        ], axis=1)
        X_others_s.append((xo - mean) / std)

    # 4 cột đích dự đoán chéo: 0=log_amt, 1=log_hours, 8=velocity, 11=merchant_risk
    target_indices = [0, 1, 8, 11]
    tr_res_list = []
    other_res_lists = [[] for _ in other_pairs]

    for tidx in target_indices:
        feat_mask = np.ones(X_tr_s.shape[1], dtype=bool)
        feat_mask[tidx] = False
        reg = Ridge(alpha=10.0)
        reg.fit(X_tr_s[:, feat_mask], X_tr_s[:, tidx])
        tr_res_list.append((X_tr_s[:, tidx] - reg.predict(X_tr_s[:, feat_mask])).astype(np.float32))
        for i, xo_s in enumerate(X_others_s):
            other_res_lists[i].append((xo_s[:, tidx] - reg.predict(xo_s[:, feat_mask])).astype(np.float32))

    return np.column_stack(tr_res_list), [np.column_stack(lst) for lst in other_res_lists]


# Bộ tự khử nhiễu hoán đổi cột (DAE) học không giám sát trên train + test
class SwapNoiseDAE(nn.Module):
    def __init__(self, in_dim: int, latent_dim: int = 64):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Linear(in_dim, 256), nn.LayerNorm(256), nn.SiLU())
        self.enc2 = nn.Sequential(nn.Linear(256, 128), nn.LayerNorm(128), nn.SiLU())
        self.enc3 = nn.Sequential(nn.Linear(128, latent_dim), nn.LayerNorm(latent_dim), nn.SiLU())
        self.dec = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.LayerNorm(128),
            nn.SiLU(),
            nn.Linear(128, 256),
            nn.LayerNorm(256),
            nn.SiLU(),
            nn.Linear(256, in_dim),
        )

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.enc3(self.enc2(self.enc1(x)))

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        z = self.encode(x)
        return self.dec(z), z


def train_swap_noise_dae(
    X_all_unlabeled: np.ndarray,
    device: torch.device,
    *,
    latent_dim: int = 64,
    swap_prob: float = 0.15,
    epochs: int = DAE_EPOCHS,
    batch_size: int = BATCH_SIZE,
    seed: int = SEED,
) -> np.ndarray:
    """Huấn luyện bộ tự khử nhiễu hoán đổi cột (Swap-Noise DAE) không giám sát trên train + test."""
    t0 = time.perf_counter()
    mean = X_all_unlabeled.mean(axis=0)
    std = np.where(X_all_unlabeled.std(axis=0) < 1e-6, 1.0, X_all_unlabeled.std(axis=0))
    X_norm = np.clip((X_all_unlabeled - mean) / std, -8.0, 8.0).astype(np.float32)

    torch.manual_seed(seed)
    model = SwapNoiseDAE(X_norm.shape[1], latent_dim=latent_dim).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1.5e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)

    X_t = torch.from_numpy(X_norm)
    n = len(X_t)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    for ep in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(n)
        loss_sum = 0.0
        for start in range(0, n, batch_size):
            idx = perm[start : start + batch_size]
            xb = X_t[idx].to(device, non_blocking=True)
            # Tạo nhiễu bằng cách hoán đổi ngẫu nhiên giá trị theo từng cột trong batch
            mask = torch.rand_like(xb) < swap_prob
            rand_row = torch.randint(0, xb.size(0), (xb.size(0),), device=device)
            xb_noisy = torch.where(mask, xb[rand_row], xb)

            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=use_amp):
                recon, _ = model(xb_noisy)
                loss = F.mse_loss(recon, xb)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            loss_sum += loss.item() * xb.size(0)
        sched.step()
        if ep == 1 or ep == epochs or ep % 4 == 0:
            print(f"  [DAE-1M] Epoch {ep:02d}/{epochs} | Recon MSE: {loss_sum / n:.5f}", flush=True)

    model.eval()
    latents = []
    with torch.inference_mode():
        for start in range(0, n, batch_size * 2):
            xb = X_t[start : start + batch_size * 2].to(device, non_blocking=True)
            latents.append(model.encode(xb).cpu().numpy())
    Z = np.concatenate(latents, axis=0).astype(np.float32)
    print(f"  [DAE-1M] Finished in {time.perf_counter()-t0:.1f}s | Latent shape: {Z.shape}", flush=True)
    return Z


# Kiến trúc TabM (BatchEnsemble) kết hợp mã hóa tuần hoàn
class PeriodicNumericalEmbeddings(nn.Module):
    """Lớp mã hóa tuần hoàn sin/cos có tần số học được cho biến số liên tục."""

    def __init__(self, n_features: int, n_frequencies: int = 16, d_embedding: int = 8, sigma: float = 0.2):
        super().__init__()
        self.coefficients = nn.Parameter(torch.randn(n_features, n_frequencies) * sigma)
        self.proj_w = nn.Parameter(torch.empty(n_features, 2 * n_frequencies, d_embedding))
        self.proj_b = nn.Parameter(torch.zeros(n_features, d_embedding))
        nn.init.kaiming_uniform_(self.proj_w, a=math.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        angles = 2.0 * math.pi * x.unsqueeze(-1) * self.coefficients.unsqueeze(0)
        periodic = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)  # [B, F, 2*n_freq]
        out = torch.einsum("bfm,fmd->bfd", periodic, self.proj_w) + self.proj_b.unsqueeze(0)
        return F.silu(out).flatten(1)  # [B, F * d_embedding]


class BatchEnsembleLinear(nn.Module):
    """Lớp tuyến tính BatchEnsemble hạng 1 cho K mạng con song song."""

    def __init__(self, in_dim: int, out_dim: int, k: int):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim, bias=False)
        self.r = nn.Parameter(torch.ones(k, in_dim))
        self.s = nn.Parameter(torch.ones(k, out_dim))
        self.b = nn.Parameter(torch.zeros(k, out_dim))
        # Khởi tạo nhiễu dấu nhẹ để các mạng con phân hóa
        nn.init.normal_(self.r, mean=1.0, std=0.1)
        nn.init.normal_(self.s, mean=1.0, std=0.1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x * self.r.unsqueeze(0)
        x = self.linear(x)
        return x * self.s.unsqueeze(0) + self.b.unsqueeze(0)


class TabMNet(nn.Module):
    """Kiến trúc TabM kết hợp mã hóa tuần hoàn và nhánh nối tắt tuyến tính cho từng mạng con."""

    def __init__(
        self,
        n_periodic_cols: int,
        n_other_cols: int,
        *,
        k: int = TABM_K,
        d_hidden: int = 256,
        n_blocks: int = 3,
        dropout: float = 0.25,
        input_dropout: float = 0.10,
    ):
        super().__init__()
        self.k = k
        self.periodic = PeriodicNumericalEmbeddings(n_periodic_cols, n_frequencies=16, d_embedding=8, sigma=0.15)
        in_dim = n_periodic_cols * 8 + n_other_cols
        self.in_drop = nn.Dropout(input_dropout)
        self.wide = BatchEnsembleLinear(in_dim, 1, k)
        self.in_proj = BatchEnsembleLinear(in_dim, d_hidden, k)
        self.blocks = nn.ModuleList([
            nn.ModuleDict({
                "norm": nn.LayerNorm(d_hidden),
                "fc1": BatchEnsembleLinear(d_hidden, d_hidden, k),
                "drop1": nn.Dropout(dropout),
                "fc2": BatchEnsembleLinear(d_hidden, d_hidden, k),
                "drop2": nn.Dropout(dropout),
            })
            for _ in range(n_blocks)
        ])
        self.head_norm = nn.LayerNorm(d_hidden)
        self.head_fc = BatchEnsembleLinear(d_hidden, 1, k)

    def forward(self, x_per: torch.Tensor, x_other: torch.Tensor) -> torch.Tensor:
        p_emb = self.periodic(x_per)
        feat = self.in_drop(torch.cat([p_emb, x_other], dim=1))  # [B, D]
        x = feat.unsqueeze(1).expand(-1, self.k, -1)  # [B, K, D]
        h = self.in_proj(x)
        for blk in self.blocks:
            u = blk["norm"](h)
            u = blk["drop1"](F.silu(blk["fc1"](u)))
            u = blk["drop2"](blk["fc2"](u))
            h = h + u
        out = self.head_fc(F.silu(self.head_norm(h))).squeeze(-1)  # [B, K]
        out = out + self.wide(x).squeeze(-1)
        return out


def tabm_bce_plus_auc_loss(logits_bk: torch.Tensor, targets_b: torch.Tensor, auc_weight: float = AUC_LOSS_WEIGHT) -> torch.Tensor:
    """Hàm mất mát kết hợp BCE theo từng mạng con và mất mát xếp hạng ghép cặp ROC-AUC."""
    # Tính BCE trên từng mạng con trước khi lấy trung bình
    k = logits_bk.size(1)
    y_bk = targets_b.unsqueeze(1).expand(-1, k)
    bce = F.binary_cross_entropy_with_logits(logits_bk, y_bk)

    if auc_weight <= 0.0:
        return bce

    # Mất mát xếp hạng ghép cặp ROC-AUC trên logit trung bình
    mean_logits = logits_bk.mean(dim=1)
    pos_mask = targets_b > 0.5
    neg_mask = ~pos_mask
    if pos_mask.any() and neg_mask.any():
        pos_l = mean_logits[pos_mask]
        neg_l = mean_logits[neg_mask]
        # Lấy mẫu con lớp âm nếu batch lớn để tiết kiệm bộ nhớ
        if neg_l.numel() > 1024:
            neg_l = neg_l[:1024]
        diff = pos_l.unsqueeze(1) - neg_l.unsqueeze(0)  # [N_pos, N_neg]
        auc_loss = F.softplus(-diff).mean()
        return bce + auc_weight * auc_loss
    return bce


def train_one_tabm_fold(
    per_tr: np.ndarray,
    oth_tr: np.ndarray,
    y_tr: np.ndarray,
    per_va: np.ndarray,
    oth_va: np.ndarray,
    y_va: np.ndarray,
    per_te: np.ndarray,
    oth_te: np.ndarray,
    device: torch.device,
    *,
    seed: int,
    label: str,
) -> tuple[np.ndarray, np.ndarray]:
    # Chuẩn hóa theo trung bình và độ lệch chuẩn của tập train
    for arrs in ((per_tr, per_va, per_te), (oth_tr, oth_va, oth_te)):
        m = arrs[0].mean(axis=0)
        s = np.where(arrs[0].std(axis=0) < 1e-6, 1.0, arrs[0].std(axis=0))
        for a in arrs:
            a -= m
            a /= s
            np.clip(a, -8.0, 8.0, out=a)

    torch.manual_seed(seed)
    model = TabMNet(per_tr.shape[1], oth_tr.shape[1], k=TABM_K, d_hidden=256, n_blocks=3, dropout=0.25, input_dropout=0.10).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1.2e-3, weight_decay=1.5e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=TABM_EPOCHS, eta_min=1e-5)

    p_tr_t, o_tr_t, y_tr_t = torch.from_numpy(per_tr), torch.from_numpy(oth_tr), torch.from_numpy(y_tr.astype(np.float32))
    p_va_t, o_va_t = torch.from_numpy(per_va), torch.from_numpy(oth_va)
    p_te_t, o_te_t = torch.from_numpy(per_te), torch.from_numpy(oth_te)

    n = len(y_tr)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    top_k: list[tuple[float, int, dict[str, torch.Tensor]]] = []
    best_auc = -1.0
    best_ep = 0

    @torch.inference_mode()
    def predict_mean_logits(pt: torch.Tensor, ot: torch.Tensor) -> np.ndarray:
        model.eval()
        out = []
        for s in range(0, len(pt), BATCH_SIZE * 2):
            pb = pt[s : s + BATCH_SIZE * 2].to(device, non_blocking=True)
            ob = ot[s : s + BATCH_SIZE * 2].to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=use_amp):
                logits_bk = model(pb, ob)
            out.append(logits_bk.float().mean(dim=1).cpu().numpy())
        return np.concatenate(out, axis=0)

    for ep in range(1, TABM_EPOCHS + 1):
        model.train()
        perm = torch.randperm(n)
        for s in range(0, n, BATCH_SIZE):
            idx = perm[s : s + BATCH_SIZE]
            pb = p_tr_t[idx].to(device, non_blocking=True)
            ob = o_tr_t[idx].to(device, non_blocking=True)
            yb = y_tr_t[idx].to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=use_amp):
                logits_bk = model(pb, ob)
                loss = tabm_bce_plus_auc_loss(logits_bk, yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
        sched.step()

        va_l = predict_mean_logits(p_va_t, o_va_t)
        auc = float(roc_auc_score(y_va, va_l))
        st = {k_: v_.detach().cpu().clone() for k_, v_ in model.state_dict().items()}
        top_k.append((auc, ep, st))
        top_k.sort(key=lambda t: t[0], reverse=True)
        top_k = top_k[:3]
        if auc > best_auc:
            best_auc = auc
            best_ep = ep
        if ep == 1 or ep == best_ep or ep % 5 == 0:
            print(f"    [{label}] Epoch {ep:02d} | Val AUC: {auc:.6f} (best={best_auc:.6f} @ ep {best_ep})", flush=True)
        if ep - best_ep >= TABM_PATIENCE:
            break

    va_snaps, te_snaps = [], []
    for _, _, st in top_k:
        model.load_state_dict(st)
        va_snaps.append(predict_mean_logits(p_va_t, o_va_t))
        te_snaps.append(predict_mean_logits(p_te_t, o_te_t))
    return np.mean(va_snaps, axis=0).astype(np.float32), np.mean(te_snaps, axis=0).astype(np.float32)


def build_fold_te_arrays(
    all_keys: dict[str, np.ndarray],
    all_freqs: dict[str, np.ndarray],
    test_keys: dict[str, np.ndarray],
    test_freqs: dict[str, np.ndarray],
    y_all: np.ndarray,
    tr_idx: np.ndarray,
    va_idx: np.ndarray,
    *,
    seed: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    y_tr = y_all[tr_idx]
    n = len(y_tr)
    tot_y = float(y_tr.sum())
    prior = tot_y / max(n, 1)
    prior_l = float(safe_logit(prior))
    inner_splits = min(5, max(2, int(min(tot_y, n - tot_y))))
    skf = StratifiedKFold(n_splits=inner_splits, shuffle=True, random_state=seed)
    inner_folds = [idx for _, idx in skf.split(np.zeros(n), y_tr)]

    tr_cols, va_cols, te_cols = [], [], []
    for name, full_k in all_keys.items():
        k_tr, k_va, k_te = full_k[tr_idx], full_k[va_idx], test_keys[name]
        uniq, inv, tot_cnt = np.unique(k_tr, return_inverse=True, return_counts=True)
        tot_sum = np.bincount(inv, weights=y_tr, minlength=uniq.size)
        m = 5.0 if uniq.size < 2000 else 15.0
        oof = np.zeros(n, dtype=np.float32)
        for vi in inner_folds:
            fi = inv[vi]
            fc = np.bincount(fi, minlength=uniq.size)
            fs = np.bincount(fi, weights=y_tr[vi], minlength=uniq.size)
            oc = tot_cnt[fi] - fc[fi]
            os_ = tot_sum[fi] - fs[fi]
            fp = (tot_y - float(y_tr[vi].sum())) / max(n - len(vi), 1)
            p = (os_ + m * fp) / (oc + m)
            oof[vi] = (safe_logit(p) - safe_logit(fp)).astype(np.float32)
        full_woe = (safe_logit((tot_sum + m * prior) / (tot_cnt + m)) - prior_l).astype(np.float32)
        tr_cols.append(oof)
        va_cols.append(_lookup_sorted(k_va, uniq, full_woe, 0.0))
        te_cols.append(_lookup_sorted(k_te, uniq, full_woe, 0.0))

    for name, full_k in all_freqs.items():
        k_tr, k_va, k_te = full_k[tr_idx], full_k[va_idx], test_freqs[name]
        uniq, inv, cnt = np.unique(k_tr, return_inverse=True, return_counts=True)
        lc = np.log1p(cnt.astype(np.float64)).astype(np.float32)
        tr_cols.append(lc[inv])
        va_cols.append(_lookup_sorted(k_va, uniq, lc, 0.0))
        te_cols.append(_lookup_sorted(k_te, uniq, lc, 0.0))

    return np.column_stack(tr_cols), np.column_stack(va_cols), np.column_stack(te_cols)


def main():
    t_start = time.perf_counter()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    data_dir = locate_data_dir()
    out_dir = locate_output_dir()
    print(f"Device: {device} | Data: {data_dir} | Output: {out_dir} | DRY_RUN={DRY_RUN}", flush=True)

    train_df = pd.read_csv(data_dir / "train.csv")
    test_df = pd.read_csv(data_dir / "test.csv")
    sample_sub = pd.read_csv(data_dir / "sample_submission.csv", usecols=[ID_COLUMN])

    if DRY_RUN:
        # Lấy lát cắt nhỏ cân bằng lớp khi chạy thử nhanh
        pos = train_df[train_df[TARGET_COLUMN] == 1].head(300)
        neg = train_df[train_df[TARGET_COLUMN] == 0].head(2700)
        train_df = pd.concat([pos, neg], ignore_index=True)
        test_df = test_df.head(500).reset_index(drop=True)
        sample_sub = pd.DataFrame({ID_COLUMN: test_df[ID_COLUMN]})

    y_all = train_df[TARGET_COLUMN].to_numpy(dtype=np.int64)
    train_ids = train_df[ID_COLUMN].to_numpy()
    test_ids = test_df[ID_COLUMN].to_numpy()
    sample_ids = pd.Index(sample_sub[ID_COLUMN])

    print(f"Train rows: {len(train_df):,} (fraud={y_all.mean():.4%}) | Test rows: {len(test_df):,}", flush=True)

    num_tr_all, dom_tr_all, bin_tr_all, oh_tr_all, vocabs = build_base_arrays(train_df)
    num_te_all, dom_te_all, bin_te_all, oh_te_all, _ = build_base_arrays(test_df, cat_vocabs=vocabs)
    all_keys, all_freqs = extract_keys(train_df)
    test_keys, test_freqs = extract_keys(test_df)

    # Huấn luyện DAE không giám sát trên toàn bộ train + test
    print("\n=== Step 1: Unsupervised Swap-Noise Denoising Autoencoder (All Rows, No Labels) ===", flush=True)
    X_unlab_tr = np.concatenate([num_tr_all, dom_tr_all, bin_tr_all, oh_tr_all], axis=1)
    X_unlab_te = np.concatenate([num_te_all, dom_te_all, bin_te_all, oh_te_all], axis=1)
    Z_all = train_swap_noise_dae(np.vstack([X_unlab_tr, X_unlab_te]), device, latent_dim=64)
    Z_tr_all = Z_all[: len(train_df)]
    Z_te_all = Z_all[len(train_df) :]

    # Khóa chia chéo StratifiedKFold 5-Fold (seed=42) khớp với mô hình cục bộ
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    oof_tabm_core = np.zeros(len(train_df), dtype=np.float64)
    test_tabm_core = np.zeros(len(test_df), dtype=np.float64)
    oof_tabm_dae = np.zeros(len(train_df), dtype=np.float64)
    test_tabm_dae = np.zeros(len(test_df), dtype=np.float64)

    for fold_id, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(y_all)), y_all)):
        print(f"\n{'=' * 72}\nFOLD {fold_id + 1}/{N_SPLITS} (Train={len(tr_idx):,} | Val={len(va_idx):,})\n{'=' * 72}", flush=True)
        y_tr, y_va = y_all[tr_idx], y_all[va_idx]

        # Tính phần dư hồi quy chéo trên tập train từng fold
        res_tr, [res_va, res_te] = fit_transform_cross_residuals(
            num_tr_all[tr_idx],
            oh_tr_all[tr_idx],
            [(num_tr_all[va_idx], oh_tr_all[va_idx]), (num_te_all, oh_te_all)],
        )
        # Tính WoE lồng 5-Fold và tần suất trên tập train từng fold
        te_tr, te_va, te_te = build_fold_te_arrays(
            all_keys, all_freqs, test_keys, test_freqs, y_all, tr_idx, va_idx, seed=SEED + fold_id * 17
        )

        # Đầu vào mã hóa tuần hoàn: 13 cột số + 4 phần dư chéo
        per_tr = np.concatenate([num_tr_all[tr_idx], res_tr], axis=1).astype(np.float32)
        per_va = np.concatenate([num_tr_all[va_idx], res_va], axis=1).astype(np.float32)
        per_te = np.concatenate([num_te_all, res_te], axis=1).astype(np.float32)

        # Mô hình A: TabM lõi
        oth_tr_a = np.concatenate([dom_tr_all[tr_idx], bin_tr_all[tr_idx], oh_tr_all[tr_idx], te_tr], axis=1).astype(np.float32)
        oth_va_a = np.concatenate([dom_tr_all[va_idx], bin_tr_all[va_idx], oh_tr_all[va_idx], te_va], axis=1).astype(np.float32)
        oth_te_a = np.concatenate([dom_te_all, bin_te_all, oh_te_all, te_te], axis=1).astype(np.float32)

        va_a, te_a = train_one_tabm_fold(
            per_tr.copy(), oth_tr_a, y_tr, per_va.copy(), oth_va_a, y_va, per_te.copy(), oth_te_a,
            device, seed=SEED + fold_id, label=f"TabM-Core F{fold_id+1}"
        )
        oof_tabm_core[va_idx] = va_a
        test_tabm_core += te_a / N_SPLITS
        print(f"  --> TabM-Core Fold {fold_id+1} Snapshot AUC: {roc_auc_score(y_va, va_a):.6f}", flush=True)

        # Mô hình B: TabM kết hợp 64 đặc trưng ẩn từ DAE
        oth_tr_b = np.concatenate([dom_tr_all[tr_idx], bin_tr_all[tr_idx], oh_tr_all[tr_idx], te_tr, Z_tr_all[tr_idx]], axis=1).astype(np.float32)
        oth_va_b = np.concatenate([dom_tr_all[va_idx], bin_tr_all[va_idx], oh_tr_all[va_idx], te_va, Z_tr_all[va_idx]], axis=1).astype(np.float32)
        oth_te_b = np.concatenate([dom_te_all, bin_te_all, oh_te_all, te_te, Z_te_all], axis=1).astype(np.float32)

        va_b, te_b = train_one_tabm_fold(
            per_tr.copy(), oth_tr_b, y_tr, per_va.copy(), oth_va_b, y_va, per_te.copy(), oth_te_b,
            device, seed=1000 + SEED + fold_id, label=f"TabM-DAE F{fold_id+1}"
        )
        oof_tabm_dae[va_idx] = va_b
        test_tabm_dae += te_b / N_SPLITS
        print(f"  --> TabM-DAE  Fold {fold_id+1} Snapshot AUC: {roc_auc_score(y_va, va_b):.6f}", flush=True)

    auc_core = float(roc_auc_score(y_all, oof_tabm_core))
    auc_dae = float(roc_auc_score(y_all, oof_tabm_dae))
    oof_blend = 0.5 * oof_tabm_core + 0.5 * oof_tabm_dae
    test_blend = 0.5 * test_tabm_core + 0.5 * test_tabm_dae
    auc_blend = float(roc_auc_score(y_all, oof_blend))
    rec_blend = float(recall_score(y_all, expit(oof_blend) >= 0.5, zero_division=0))

    print(f"\n{'=' * 72}")
    print(f"TabM-Core {N_SPLITS}-Fold OOF AUC : {auc_core:.6f}")
    print(f"TabM-DAE  {N_SPLITS}-Fold OOF AUC : {auc_dae:.6f}")
    print(f"TabM Blend (50/50) OOF AUC : {auc_blend:.6f} | Recall@0.5: {rec_blend:.6f}")
    print(f"Total elapsed: {(time.perf_counter() - t_start) / 60:.1f} min")
    print(f"{'=' * 72}", flush=True)

    # Lưu tệp OOF kèm cột id để tổ hợp với các mô hình cục bộ
    pd.DataFrame({
        ID_COLUMN: train_ids,
        TARGET_COLUMN: y_all,
        "pred_prob": expit(oof_blend),
        "logit_core": oof_tabm_core,
        "logit_dae": oof_tabm_dae,
    }).to_csv(out_dir / "oof_kaggle_tabm_5fold.csv", index=False)

    sub_df = pd.DataFrame({ID_COLUMN: sample_ids})
    sub_df[TARGET_COLUMN] = pd.Series(expit(test_blend), index=test_ids).reindex(sample_ids).to_numpy()
    sub_df.to_csv(out_dir / "submission.csv", index=False)
    sub_df.to_csv(out_dir / "sub_kaggle_tabm_5fold.csv", index=False)
    print(f"Saved {out_dir / 'submission.csv'} and {out_dir / 'oof_kaggle_tabm_5fold.csv'}", flush=True)




In [ ]:
main()
